# Agent walkthrough

Runs the graph from `anthology_rag.agent` one node at a time: `parse_query` (LLM) → `retrieve` (vector search) → `respond` (LLM).

Requires a built database (`uv run build-db`) and the LLM server from `LLM_BASE_URL`. Run with the project root as working directory; settings come from `.env`.

In [1]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

from anthology_rag import agent
from anthology_rag.config import (
    LLM_BASE_URL,
    LLM_MODEL,
    LLM_TEMPERATURE,
    NUM_CANDIDATES_WITH_AUTHORS,
    NUM_RESULTS,
    PERSIST_DIR,
)
from anthology_rag.db import get_embeddings, get_store

llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    api_key="none",
    model=LLM_MODEL,
    temperature=LLM_TEMPERATURE,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
store = get_store(PERSIST_DIR, get_embeddings())
print(f"{store._collection.count()} papers in {PERSIST_DIR.resolve()}")

/home/simeon/Dokumente/Code/Repos/anthology_rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9123.53it/s]


10000 papers in /home/simeon/Dokumente/Code/Repos/anthology_rag/chroma_db


## The conversation

Edit `messages` to try other requests. Follow-ups are shown with the compiled graph at the end.

In [7]:
messages = [
    HumanMessage("Find papers about neural machine translation")
]
agent._history(messages)  # what the LLM sees (trimmed to MAX_HISTORY_MESSAGES)

[HumanMessage(content='Find papers about neural machine translation', additional_kwargs={}, response_metadata={})]

## Node 1: `parse_query` (LLM)

The LLM turns the history into a structured `Filters` object.

In [8]:
print(agent.PARSE_PROMPT)
parser = llm.with_structured_output(agent.Filters)
history = agent._history(messages)
print("History:  ", history)
raw_filters = parser.invoke([SystemMessage(agent.PARSE_PROMPT), *history])
print("LLM output:  ", raw_filters)

# Same guard as in the graph: drops authors/years/venues the user never wrote.
filters = agent._only_requested(raw_filters, history)
print("After guard: ", filters)

Turn the user's latest request for ACL Anthology papers into a standalone search. Use the conversation to resolve follow-ups such as 'only from 2023' or 'by the same authors'. Keep the previous constraints unless the user changes or drops them. Venues are lowercase anthology venue IDs, e.g. acl, emnlp, naacl, coling, lrec. Only set authors, years or venues that the user explicitly asked for in the conversation; never guess or infer them. Leave them empty otherwise.
History:   [HumanMessage(content='Find papers about neural machine translation', additional_kwargs={}, response_metadata={})]
LLM output:   topic='Neural Machine Translation' authors=[] year_from=None year_to=None venues=[]
After guard:  topic='Neural Machine Translation' authors=[] year_from=None year_to=None venues=[]


## Node 2: `retrieve` (no LLM)

Filters become a Chroma `where` clause; the vector search uses only `filters.topic`. With authors, more candidates are fetched and then filtered by name.

In [9]:
where = agent._where(filters)
k = NUM_CANDIDATES_WITH_AUTHORS if filters.authors else NUM_RESULTS
print("where:", where)
print("k:", k)

where: None
k: 10


In [10]:
# cosine distance: lower = closer
scored = store.similarity_search_with_score(filters.topic, k=k, filter=where)
print(f"{len(scored)} candidates from the vector search")

docs = [d for d, _ in scored]
for author in filters.authors:
    docs = [d for d in docs if author.lower() in d.metadata["authors"].lower()]
docs = docs[:NUM_RESULTS]

kept = {d.id for d in docs}
for d, dist in scored[:50]:
    m = d.metadata
    print(
        f"{'*' if d.id in kept else ' '} {dist:.3f}  {m['year']}  {m['venues']:<15} "
        f"{m['title'][:70]}  [{m['authors'][:40]}]"
    )
print("\n* = passed the author filter and is sent to the LLM")

10 candidates from the vector search
* 0.137  2026  eamt            From Insights to Impact: Actionable Interpretability for Neural Machin  [Gabriele Sarti]
* 0.164  2026  arabicnlp       From Subwords to Pixels: Evaluating Text Representations for Arabic Ma  [Abdelrahman Shaheen; Mohamed Abbas; Tere]
* 0.171  2026  eamt            Augmenting Text to Increase Translation Difficulty  [William Kalikman; Simon Sukup; Michal Te]
* 0.178  2026  findings        Learning to Translate by Translating: Stabilizing the Dual Loop via Se  [Kui Liu; Mingming Yin; Zuoli Tang; Zihao]
* 0.180  2026  eamt            Improving Retrieval-Augmented Neural Machine Translation with Monoling  [Maxime Bouthors; Josep Crego; Dakun Zhan]
* 0.180  2026  eacl            Thesis proposal: Are We Losing Textual Diversity to Natural Language P  [Josef Jon; Ondřej Bojar]
* 0.185  2026  americasnlp,ws  A Systematic Comparison of Parameter-Efficient Fine-Tuning Techniques   [Drew Stackhouse; Justin Debenedetto]
* 0.185  

## Node 3: `respond` (LLM)

The retrieved papers are put into the system prompt; only the reply is added to the history.

In [11]:
if docs:
    papers = "\n".join(agent._format_paper(d) for d in docs)
    prompt = SystemMessage(agent.RESPOND_PROMPT.format(papers=papers))
    print(prompt.content)
else:
    print("No docs: respond returns a fixed message without calling the LLM.")

You help users find papers in the ACL Anthology. Answer using ONLY the papers below. List the relevant ones as a numbered list with title, authors, venue, year and URL. Do not invent papers.

Retrieved papers:
- From Insights to Impact: Actionable Interpretability for Neural Machine Translation | authors: Gabriele Sarti | venue: eamt | year: 2026 | url: https://aclanthology.org/2026.eamt-1.1/
  abstract: 
- From Subwords to Pixels: Evaluating Text Representations for Arabic Machine Translation | authors: Abdelrahman Shaheen; Mohamed Abbas; Tereza Truckova; Yova Kementchedjhieva; Bashar Alhafni | venue: arabicnlp | year: 2026 | url: https://aclanthology.org/2026.arabicnlp-main.19/
  abstract: Subword tokenization underpins modern neural NLP but can be suboptimal for morphologically rich languages with substantial orthographic variation. In this paper, we compare vocabulary-based and vocabulary-free representations for Arabic-to-English machine translation, including subword, byte-level,

In [12]:
if docs:
    reply = llm.invoke([prompt, *agent._history(messages)])
else:
    reply = AIMessage("No papers matched your request.")
print(reply.content)

KeyboardInterrupt: 

## Same thing through the compiled graph

`stream_mode="updates"` shows what each node writes to the state. The checkpointer keeps the history per `thread_id`, so the second call is a follow-up that has to be resolved from the conversation.

In [ ]:
import uuid

graph = agent.build_graph(llm, store)
config = {"configurable": {"thread_id": str(uuid.uuid4())}}


def ask(question: str) -> None:
    for update in graph.stream({"messages": [("user", question)]}, config, stream_mode="updates"):
        for node, state in update.items():
            print(f"--- {node} ---")
            if "filters" in state:
                print(state["filters"])
            if "docs" in state:
                for d in state["docs"]:
                    print(f"  {d.metadata['year']}  {d.metadata['title']}")
            if "messages" in state:
                print(state["messages"][-1].content)


ask("Find papers about speech recognition with transformers")

In [ ]:
# Follow-up in the same thread: previous constraints should carry over, with the year added.
ask("Only from 2023")

In [ ]:
# Stored history for this thread (retrieved papers are not part of it).
for m in graph.get_state(config).values["messages"]:
    print(f"[{m.type}] {m.content[:120]!r}")

## Live token streaming

`stream_mode="messages"` yields every token chunk the LLM produces, together with the node that produced it. `parse_query` uses structured output, so its tokens usually arrive as tool-call argument fragments (or JSON text) rather than plain content; both are shown. `retrieve` makes no LLM call, so it never appears here.

In [13]:
import uuid

graph = agent.build_graph(llm, store)


def ask_streaming(question: str, thread_id: str | None = None) -> None:
    cfg = {"configurable": {"thread_id": thread_id or str(uuid.uuid4())}}
    current_node = None
    for chunk, meta in graph.stream(
        {"messages": [("user", question)]}, cfg, stream_mode="messages"
    ):
        node = meta["langgraph_node"]
        # Structured output arrives as tool-call argument fragments, plain output as content.
        text = chunk.content or "".join(c["args"] or "" for c in chunk.tool_call_chunks)
        if not isinstance(text, str) or not text:
            continue
        if node != current_node:
            print(f"\n\n--- {node} ---")
            current_node = node
        print(text, end="", flush=True)
    print()


ask_streaming("Find papers about speech recognition")



--- parse_query ---
{
  "topic": "Speech Recognition",
  "authors": [],
  "year_from": null,
  "year_to": null,
  "venues": []
}

/home/simeon/Dokumente/Code/Repos/anthology_rag/.venv/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Filters(topic='Speech Rec...year_to=None, venues=[]), input_type=Filters])
  return self.__pydantic_serializer__.to_python(




--- respond ---
Based on the provided list, here are the relevant papers about speech recognition:

1. Vietnamese Automatic Speech Recognition: A Revisit | authors: Thi Vu; Linh The Nguyen; Dat Quoc Nguyen | venue: findings | year: 2026 | url: https://aclanthology.org/2026.findings-eacl.345/
2. Phonologically-aware Automatic Speech Recognition Evaluation of Low-Resource Languages: The Case of Basque Dialects | authors: Christoforos Souganidis; Asier Herranz; Ibon Saratxaga; Eva Navas; Inma Hernaez | venue: dialres,ws | year: 2026 | url: https://aclanthology.org/2026.dialres-1.5/
3. Listening Like Humans: Semantics-Guided Noise-Robust Multimodal Speech Recognition | authors: Yan Fang; Jun Chen; Yian Yao; Shuxin Zhong; Min Sun; Kaishun Wu | venue: acl | year: 2026 | url: https://aclanthology.org/2026.acl-long.730/
4. PRiSM: Benchmarking Phone Realization in Speech Models | authors: Shikhar Bharadwaj; Chin-Jou Li; Yoonjae Kim; Kwanghee Choi; Eunjung Yeo; Ryan Soh-Eun Shim; Hanyu Zhou; B